1.Imports and setup

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import void

In [0]:
spark.sql("use catalog novacart_adb")


DataFrame[]

In [0]:
spark.sql("create schema if not exists bronze_schema")

DataFrame[]

2.Bronze control table

In [0]:
spark.sql("""
          create table if not exists novacart_adb.bronze_schema.ingestion_control 
          (
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
          )
          using delta
          """)

DataFrame[]

3.Source table configuration

In [0]:
import uuid

tables_config={
    "orders": {  "ts_col": "updated_at", "pk_col": "order_id"},
    "products": {"ts_col": "updated_at", "pk_col": "product_id"},
    "payments": {"ts_col": "processed_at", "pk_col": "payment_id"}
}
bronze_run_id = str(uuid.uuid4())
print("Current Bronze Run ID: ", bronze_run_id)

Current Bronze Run ID:  e71564fd-bf16-4ed6-9e7d-4c176ab955eb


4.helper function

In [0]:
def get_last_successful_watermark(table_name:str):
  ctrl =(
    spark.table("novacart_adb.bronze_schema.ingestion_control")
    .filter(
      (F.col("layer")=="bronze") 
      & (F.col("table_name")==table_name)&
      (F.col("run_status")=="success")
    )
    .orderBy(F.col("updated_at").desc())
    .limit(1)
  )
  rows=ctrl.collect()
  if not rows:
    return None,None

  return rows[0]["last_successful_ts"],rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df = spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None else None,
                run_id,
                int(rows_written),
                "success",
                datetime.now()
            )],
        schema="""
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )
    dt=DeltaTable.forName(spark, "novacart_adb.bronze_schema.ingestion_control")
    (dt.alias("t").merge(control_df.alias("source"), "t.table_name = source.table_name and t.layer = source.layer")
        .whenMatchedUpdate(set={
        "ts_col":"source.ts_col",
        "pk_col":"source.pk_col",
        "last_successful_ts":"source.last_successful_ts",
        "last_successful_pk":"source.last_successful_pk",
        "last_run_id":"source.last_run_id",
        "rows_written":"source.rows_written",
        "run_status":"source.run_status",
        "updated_at":"source.updated_at"
    })
    
    .whenNotMatchedInsertAll()
    .execute()
    )
    

5.bronze incremental load loop

In [0]:
!pip install uuid
import uuid

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
for table_name,config in tables_config.items():
    ts_col=config["ts_col"]
    pk_col=config["pk_col"]
    source_table=f"novacart_sql_connection_catalog.dbo.{table_name}"
    target_table=f"novacart_adb.bronze_schema.{table_name}"
    last_successful_ts,last_successful_pk=get_last_successful_watermark(table_name)
    print(f"\n=== Processing table {table_name}=== ")
    print(f"Last successful ts: {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")
    source_df=spark.table(source_table).withColumn(ts_col,F.col(ts_col).cast("timestamp"))
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) & (F.col(pk_col).cast("bigint") > F.lit(last_successful_pk))
            )
        )
    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at",F.current_timestamp())
        .withColumn("bronze_run_id",F.lit(bronze_run_id))
        .withColumn("bronze_source_table",F.lit(source_table))
    )
    row_count=rows_to_load.count()

    print(f"{table_name} rows_to_load={row_count}")

    if row_count==0:
        print(f"No new rows to load for {table_name}.")
        upsert_bronze_control(table_name,ts_col,pk_col,last_successful_ts,last_successful_pk,row_count,bronze_run_id)
        continue
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)
    max_pk =(
        rows_to_load
        .agg(F.max(pk_col).alias("max_pk"))
        .collect()[0]["max_pk"]
    )
    upsert_bronze_control(table_name,ts_col,pk_col,last_successful_ts,max_pk,row_count,bronze_run_id)
    print(f"Wrote {row_count} rows to {target_table}")


=== Processing table orders=== 
Last successful ts: None
Last successful pk: 200500


---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-5158399557268803>, line 26
     14     rows_to_load = source_df.filter(
     15         (F.col(ts_col) > F.lit(last_successful_ts)) |
     16         (
     17             (F.col(ts_col) == F.lit(last_successful_ts)) & (F.col(pk_col).cast("bigint") > F.lit(last_successful_pk))
     18         )
     19     )
     20 rows_to_load = (
     21     rows_to_load
     22     .withColumn("bronze_ingested_at",F.current_timestamp())
     23     .withColumn("bronze_run_id",F.lit(bronze_run_id))
     24     .withColumn("bronze_source_table",F.lit(source_table))
     25 )
---> 26 row_count=rows_to_load.count()
     28 print(f"{table_name} rows_to_load={row_count}")
     30 if row_count==0:

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int

6.Quick validation


In [0]:
print("Orders Bronze Count: ",spark.sql("select count(*) from novacart_adb.bronze_schema.orders").collect()[0][0])
print("Products Bronze Count: ",spark.sql("select count(*) from novacart_adb.bronze_schema.products").collect()[0][0])
print("Orders Bronze Count: ",spark.sql("select count(*) from novacart_adb.bronze_schema.orders").collect()[0][0])  

display(spark.sql("select * from novacart_adb.bronze_schema.ingestion_control").orderBy("table_name"))

Orders Bronze Count:  1500
Products Bronze Count:  240
Orders Bronze Count:  1500


layer,table_name,ts_col,pk_col,last_successful_ts,last_successful_pk,last_run_id,rows_written,run_status,updated_at
bronze,orders,updated_at,order_id,null,200500,0657dba9-fdf2-467c-8184-a916c8ea5bd3,500,success,2026-03-30T19:08:19.599Z
bronze,payments,processed_at,payment_id,null,900430,0657dba9-fdf2-467c-8184-a916c8ea5bd3,430,success,2026-03-30T19:08:33.999Z
bronze,products,updated_at,product_id,null,1120,0657dba9-fdf2-467c-8184-a916c8ea5bd3,120,success,2026-03-30T19:08:26.707Z
